# Preparación SQL · FI Videojuegos · JMCC
SQL nativo, sin Python ni volúmenes. Requiere Runtime 16.3 o superior (recomendado 16.4 LTS+).
La credencial y los contenedores deben existir. No crea un SQL Warehouse.

In [0]:
CREATE WIDGET TEXT storageName DEFAULT 'adlsmardata1412jmvcc';
CREATE WIDGET TEXT catalogo DEFAULT 'fi_videojuegos';
CREATE WIDGET TEXT storageCredential DEFAULT 'credential';
CREATE WIDGET TEXT container_metastore DEFAULT 'metastore';
CREATE WIDGET TEXT carpeta_proyecto DEFAULT 'fi_videojuegos_jmcc';
CREATE WIDGET TEXT raw_subdir DEFAULT '';
CREATE WIDGET TEXT aplicar_grants DEFAULT 'false';


In [0]:
SELECT
  assert_true(:catalogo RLIKE '^[a-z][a-z0-9_]*$', 'catalogo debe ser un identificador simple en minusculas'),
  assert_true(:storageName RLIKE '^[a-z0-9]{3,24}$', 'storageName invalido'),
  assert_true(:storageCredential RLIKE '^[A-Za-z0-9_-]+$', 'storageCredential invalido'),
  assert_true(:container_metastore RLIKE '^[a-z0-9][a-z0-9-]*$', 'Contenedor metastore invalido'),
  assert_true(:carpeta_proyecto RLIKE '^[a-z0-9_/-]+$' AND instr(:carpeta_proyecto, '..') = 0
    AND :carpeta_proyecto NOT LIKE '/%' AND :carpeta_proyecto NOT LIKE '%/', 'Carpeta de proyecto invalida'),
  assert_true(:raw_subdir = '' OR (:raw_subdir RLIKE '^[A-Za-z0-9_/-]+$'
    AND instr(:raw_subdir, '..') = 0 AND :raw_subdir NOT LIKE '/%' AND :raw_subdir NOT LIKE '%/'), 'raw_subdir invalida');
SHOW EXTERNAL LOCATIONS;

## Ubicaciones externas
Reutiliza las rutas ya autorizadas cuando existan; crea nombres propios solo donde no hay cobertura visible.

In [0]:
-- Reutilizar ubicaciones visibles que YA cubran la ruta: no crear rutas superpuestas.
-- Si no hay cobertura se crea una ubicacion propia con nombre fi_videojuegos_jmcc_*.
BEGIN
  DECLARE v_url STRING;
  DECLARE v_name STRING;
  DECLARE v_stmt STRING;
  DECLARE v_cover BIGINT;
  DECLARE v_child BIGINT;
  FOR r AS SELECT * FROM VALUES ('metastore'), ('raw'), ('bronze'), ('silver'), ('golden') AS capas(capa) DO
    SET v_url = concat('abfss://', CASE WHEN r.capa = 'metastore' THEN :container_metastore ELSE r.capa END,
      '@', :storageName, '.dfs.core.windows.net/',
      CASE WHEN r.capa = 'raw' THEN :raw_subdir ELSE :carpeta_proyecto END);
    SET v_url = concat(rtrim('/', v_url), '/');
    SET v_name = concat(:catalogo, '_jmcc_', r.capa);
    SELECT assert_true(count(*) = 0, 'Nombre de ubicacion existente con otra URL; revisar antes de continuar')
    FROM system.information_schema.external_locations
    WHERE external_location_name = v_name AND concat(rtrim('/', url), '/') <> v_url;
    SET v_cover = (SELECT count(*) FROM system.information_schema.external_locations
      WHERE startswith(v_url, concat(rtrim('/', url), '/')));
    IF v_cover = 0 THEN
      SET v_child = (SELECT count(*) FROM system.information_schema.external_locations
        WHERE startswith(concat(rtrim('/', url), '/'), v_url));
      SELECT assert_true(v_child = 0, 'Hay una ubicacion descendiente: ajustar alcance, no crear una superpuesta');
      SET v_stmt = concat(
  'CREATE EXTERNAL LOCATION IF NOT EXISTS `', v_name,
  '` URL ', chr(39), v_url, chr(39),
  ' WITH (STORAGE CREDENTIAL `', :storageCredential, '`)',
  ' COMMENT ', chr(39), 'Proyecto FI videojuegos JMCC', chr(39)
);

EXECUTE IMMEDIATE v_stmt;
    END IF;
  END FOR;
END;

## Catálogo y esquemas propios

In [0]:
BEGIN
  DECLARE v_stmt STRING;

  SET v_stmt = concat(
    'CREATE CATALOG IF NOT EXISTS `', :catalogo,
    '` MANAGED LOCATION ', chr(39),
    'abfss://', :container_metastore, '@', :storageName,
    '.dfs.core.windows.net/', :carpeta_proyecto, '/catalogo',
    chr(39)
  );

  EXECUTE IMMEDIATE v_stmt;

  FOR r AS
    SELECT *
    FROM VALUES ('raw'), ('bronze'), ('silver'), ('golden')
      AS capas(capa)
  DO
    SET v_stmt = concat(
      'CREATE SCHEMA IF NOT EXISTS `',
      :catalogo, '`.`', r.capa, '`'
    );

    EXECUTE IMMEDIATE v_stmt;
  END FOR;
END;

## Verificar los cuatro CSV de raw
Se comprueba la huella del archivo exacto que se usaba en el proyecto. Si cambias las fuentes, hay que revisar las huellas y los conteos.

In [0]:
SELECT assert_true(count(*) = 1 AND min(sha2(content, 256)) = '9130cb5af6d344b0af4553739da7a16328e528f4e0f2342e1c6f344e5845f705',
  'Archivo no coincide con la fuente validada: nintendo.csv') AS comprobacion
FROM read_files(concat('abfss://raw@', :storageName, '.dfs.core.windows.net/',
  CASE WHEN :raw_subdir = '' THEN '' ELSE concat(:raw_subdir, '/') END, 'nintendo.csv'), format => 'binaryFile');

In [0]:
SELECT assert_true(count(*) = 1 AND min(sha2(content, 256)) = '08292466a360602b1ed478fa59c48e79b4b6d132f3d598f543ef602eb4604c9e',
  'Archivo no coincide con la fuente validada: playstation.csv') AS comprobacion
FROM read_files(concat('abfss://raw@', :storageName, '.dfs.core.windows.net/',
  CASE WHEN :raw_subdir = '' THEN '' ELSE concat(:raw_subdir, '/') END, 'playstation.csv'), format => 'binaryFile');

In [0]:
SELECT assert_true(count(*) = 1 AND min(sha2(content, 256)) = 'e2076095ffcae2a92dbc6de6ecbd54455ee034bb38da550b2dc385e9265d4ebe',
  'Archivo no coincide con la fuente validada: vgsales.csv') AS comprobacion
FROM read_files(concat('abfss://raw@', :storageName, '.dfs.core.windows.net/',
  CASE WHEN :raw_subdir = '' THEN '' ELSE concat(:raw_subdir, '/') END, 'vgsales.csv'), format => 'binaryFile');

In [0]:
SELECT assert_true(count(*) = 1 AND min(sha2(content, 256)) = '85300b4b4765c0f502fb20e958079ac2365cc39461d23b200066f1411aef2219',
  'Archivo no coincide con la fuente validada: xbox.csv') AS comprobacion
FROM read_files(concat('abfss://raw@', :storageName, '.dfs.core.windows.net/',
  CASE WHEN :raw_subdir = '' THEN '' ELSE concat(:raw_subdir, '/') END, 'xbox.csv'), format => 'binaryFile');

## Nueve tablas externas Delta
Cuatro Bronze, dos Silver y tres Golden. Los esquemas se tomaron de las fuentes y de los resultados del DBC original.

In [0]:
BEGIN
  DECLARE v_stmt STRING;
  SET v_stmt = concat('CREATE TABLE IF NOT EXISTS `', :catalogo, '`.`bronze`.`fi_br_nintendo` (
  `game_title` STRING,
  `normalized_title` STRING,
  `platform` STRING,
  `platform_original` STRING,
  `platform_vgsales_code` STRING,
  `platform_cleaning_note` STRING,
  `release_date_original` STRING,
  `release_date` STRING,
  `release_year` STRING,
  `release_month` STRING,
  `release_month_name_es` STRING,
  `release_day` STRING,
  `release_quarter` STRING,
  `date_status` STRING,
  `meta_score` STRING,
  `user_score` STRING,
  `esrb_rating` STRING,
  `link` STRING,
  `developer_count` STRING,
  `developer_1` STRING,
  `developer_2` STRING,
  `developer_3` STRING,
  `developer_4` STRING,
  `developers_original` STRING,
  `genre_count` STRING,
  `genre_1` STRING,
  `genre_2` STRING,
  `genre_3` STRING,
  `genre_4` STRING,
  `genre_5` STRING,
  `genre_6` STRING,
  `genre_7` STRING,
  `genre_8` STRING,
  `genre_9` STRING,
  `genre_10` STRING,
  `genres_original` STRING,
  `join_key_vgsales` STRING,
  `fi_fecha_ingesta` TIMESTAMP,
  `fi_archivo_origen` STRING
) USING DELTA LOCATION ',
    chr(39),
    'abfss://bronze@', :storageName,
    '.dfs.core.windows.net/', :carpeta_proyecto,
    '/fi_br_nintendo',
    chr(39)
  );

  EXECUTE IMMEDIATE v_stmt;
END;

In [0]:
BEGIN
  DECLARE v_stmt STRING;
  SET v_stmt = concat('CREATE TABLE IF NOT EXISTS `', :catalogo, '`.`bronze`.`fi_br_playstation` (
  `id_igdb` STRING,
  `nombre_videojuego` STRING,
  `fecha_primer_lanzamiento` STRING,
  `anio_primer_lanzamiento` STRING,
  `id_plataforma_igdb` STRING,
  `plataforma` STRING,
  `codigo_plataforma_vgsales` STRING,
  `coincide_codigo_vgsales` STRING,
  `titulo_normalizado` STRING,
  `clave_join_nombre_plataforma` STRING,
  `clave_join_nombre_anio_plataforma` STRING,
  `todas_las_plataformas` STRING,
  `ids_todas_las_plataformas` STRING,
  `generos` STRING,
  `ids_generos` STRING,
  `calificacion_igdb_0_100` STRING,
  `puntuacion_hltb` STRING,
  `cantidad_resenas_hltb` STRING,
  `personas_encuestadas_hltb` STRING,
  `horas_historia_principal` STRING,
  `horas_historia_mas_extras` STRING,
  `horas_completista_100` STRING,
  `resumen_original` STRING,
  `historia_original` STRING,
  `fi_fecha_ingesta` TIMESTAMP,
  `fi_archivo_origen` STRING
) USING DELTA LOCATION ',
    chr(39),
    'abfss://bronze@', :storageName,
    '.dfs.core.windows.net/', :carpeta_proyecto,
    '/fi_br_playstation',
    chr(39)
  );

  EXECUTE IMMEDIATE v_stmt;
END;

In [0]:
BEGIN
  DECLARE v_stmt STRING;
  SET v_stmt = concat(
    'CREATE TABLE IF NOT EXISTS `',
    :catalogo,
    '`.`bronze`.`fi_br_vgsales` (
      `Rank` STRING,
      `Name` STRING,
      `Platform` STRING,
      `Year` STRING,
      `Genre` STRING,
      `Publisher` STRING,
      `NA_Sales` STRING,
      `EU_Sales` STRING,
      `JP_Sales` STRING,
      `Other_Sales` STRING,
      `Global_Sales` STRING,
      fi_fecha_ingesta TIMESTAMP,
      fi_archivo_origen STRING
    ) USING DELTA LOCATION ',
    chr(39),
    'abfss://bronze@', :storageName,
    '.dfs.core.windows.net/', :carpeta_proyecto,
    '/fi_br_vgsales',
    chr(39)
  );

  EXECUTE IMMEDIATE v_stmt;
END;

In [0]:
BEGIN
  DECLARE v_stmt STRING;

  SET v_stmt = concat(
    'CREATE TABLE IF NOT EXISTS `',
    :catalogo,
    '`.`bronze`.`fi_br_xbox` (
      id_igdb STRING,
      nombre_videojuego STRING,
      fecha_primer_lanzamiento STRING,
      anio_primer_lanzamiento STRING,
      id_plataforma_igdb STRING,
      plataforma STRING,
      codigo_plataforma_vgsales STRING,
      coincide_codigo_vgsales STRING,
      titulo_normalizado STRING,
      clave_join_nombre_plataforma STRING,
      clave_join_nombre_anio_plataforma STRING,
      todas_las_plataformas STRING,
      ids_todas_las_plataformas STRING,
      generos STRING,
      ids_generos STRING,
      calificacion_igdb_0_100 STRING,
      puntuacion_hltb STRING,
      cantidad_resenas_hltb STRING,
      personas_encuestadas_hltb STRING,
      horas_historia_principal STRING,
      horas_historia_mas_extras STRING,
      horas_completista_100 STRING,
      resumen_original STRING,
      historia_original STRING,
      fi_fecha_ingesta TIMESTAMP,
      fi_archivo_origen STRING
    ) USING DELTA LOCATION ',
    chr(39),
    'abfss://bronze@', :storageName,
    '.dfs.core.windows.net/', :carpeta_proyecto,
    '/fi_br_xbox',
    chr(39)
  );

  EXECUTE IMMEDIATE v_stmt;
END;

In [0]:
BEGIN
  DECLARE v_stmt STRING;

  SET v_stmt = concat(
    'CREATE TABLE IF NOT EXISTS `',
    :catalogo,
    '`.`silver`.`fi_sl_metadatos` (
      fuente STRING,
      familia STRING,
      titulo_meta STRING,
      nombre_plataforma STRING,
      plataforma STRING,
      anio_meta INT,
      fecha_meta DATE,
      meta_score DOUBLE,
      user_score DOUBLE,
      desarrollador STRING,
      generos_meta STRING,
      id_igdb BIGINT,
      score_igdb DOUBLE,
      score_hltb DOUBLE,
      horas_principal DOUBLE,
      horas_extras DOUBLE,
      horas_completista DOUBLE,
      fi_archivo_origen STRING,
      hash_origen STRING,
      titulo_clave STRING,
      apto_join BOOLEAN,
      score_fuera_rango BOOLEAN
    ) USING DELTA LOCATION ',
    chr(39),
    'abfss://silver@', :storageName,
    '.dfs.core.windows.net/', :carpeta_proyecto,
    '/fi_sl_metadatos',
    chr(39)
  );

  EXECUTE IMMEDIATE v_stmt;
END;

In [0]:
BEGIN
  DECLARE v_stmt STRING;

  SET v_stmt = concat(
    'CREATE TABLE IF NOT EXISTS `',
    :catalogo,
    '`.`silver`.`fi_sl_ventas_enriquecidas` (
      rank_venta INT,
      titulo STRING,
      plataforma STRING,
      anio INT,
      genero STRING,
      editorial STRING,
      ventas_na DECIMAL(14,3),
      ventas_eu DECIMAL(14,3),
      ventas_jp DECIMAL(14,3),
      ventas_otras DECIMAL(14,3),
      ventas_globales DECIMAL(14,3),
      fi_archivo_origen STRING,
      fi_fecha_ingesta TIMESTAMP,
      titulo_clave STRING,
      familia STRING,
      venta_valida BOOLEAN,
      diferencia_regiones DECIMAL(18,3),
      regiones_conciliadas BOOLEAN,
      fuente STRING,
      titulo_meta STRING,
      nombre_plataforma STRING,
      anio_meta INT,
      fecha_meta DATE,
      meta_score DOUBLE,
      user_score DOUBLE,
      desarrollador STRING,
      generos_meta STRING,
      id_igdb BIGINT,
      score_igdb DOUBLE,
      score_hltb DOUBLE,
      horas_principal DOUBLE,
      horas_extras DOUBLE,
      horas_completista DOUBLE,
      hash_origen STRING,
      fi_num_candidatos BIGINT,
      confianza_join STRING
    ) USING DELTA LOCATION ',
    chr(39),
    'abfss://silver@', :storageName,
    '.dfs.core.windows.net/', :carpeta_proyecto,
    '/fi_sl_ventas_enriquecidas',
    chr(39)
  );

  EXECUTE IMMEDIATE v_stmt;
END;

In [0]:
BEGIN
  DECLARE v_stmt STRING;

  SET v_stmt = concat(
    'CREATE TABLE IF NOT EXISTS `',
    :catalogo,
    '`.`golden`.`fi_gd_hechos_ventas` (
      rank_venta INT,
      titulo STRING,
      plataforma STRING,
      anio INT,
      genero STRING,
      editorial STRING,
      ventas_na DECIMAL(14,3),
      ventas_eu DECIMAL(14,3),
      ventas_jp DECIMAL(14,3),
      ventas_otras DECIMAL(14,3),
      ventas_globales DECIMAL(14,3),
      fi_archivo_origen STRING,
      fi_fecha_ingesta TIMESTAMP,
      titulo_clave STRING,
      familia STRING,
      venta_valida BOOLEAN,
      diferencia_regiones DECIMAL(18,3),
      regiones_conciliadas BOOLEAN,
      fuente STRING,
      titulo_meta STRING,
      nombre_plataforma STRING,
      anio_meta INT,
      fecha_meta DATE,
      meta_score DOUBLE,
      user_score DOUBLE,
      desarrollador STRING,
      generos_meta STRING,
      id_igdb BIGINT,
      score_igdb DOUBLE,
      score_hltb DOUBLE,
      horas_principal DOUBLE,
      horas_extras DOUBLE,
      horas_completista DOUBLE,
      hash_origen STRING,
      fi_num_candidatos BIGINT,
      confianza_join STRING,
      decada INT,
      con_metadatos BOOLEAN
    ) USING DELTA LOCATION ',
    chr(39),
    'abfss://golden@', :storageName,
    '.dfs.core.windows.net/', :carpeta_proyecto,
    '/fi_gd_hechos_ventas',
    chr(39)
  );

  EXECUTE IMMEDIATE v_stmt;
END;

In [0]:
BEGIN
  DECLARE v_stmt STRING;

  SET v_stmt = concat(
    'CREATE TABLE IF NOT EXISTS `',
    :catalogo,
    '`.`golden`.`fi_gd_ranking_familia_anio` (
      anio INT,
      familia STRING,
      registros_juego_plataforma BIGINT,
      ventas_globales_millones DECIMAL(24,3),
      puesto INT,
      total_tres_familias_millones DECIMAL(34,3),
      participacion_pct DECIMAL(31,2)
    ) USING DELTA LOCATION ',
    chr(39),
    'abfss://golden@', :storageName,
    '.dfs.core.windows.net/', :carpeta_proyecto,
    '/fi_gd_ranking_familia_anio',
    chr(39)
  );

  EXECUTE IMMEDIATE v_stmt;
END;

In [0]:
BEGIN
  DECLARE v_stmt STRING;

  SET v_stmt = concat(
    'CREATE TABLE IF NOT EXISTS `',
    :catalogo,
    '`.`golden`.`fi_gd_mejor_juego_familia_decada` (
      inicio_decada INT,
      familia STRING,
      decada STRING,
      mejor_juego STRING,
      plataforma STRING,
      ventas_millones DECIMAL(14,3),
      segundo_juego STRING,
      plataforma_segundo STRING,
      ventas_segundo_millones DECIMAL(14,3),
      supero_al_segundo_pct DOUBLE
    ) USING DELTA LOCATION ',
    chr(39),
    'abfss://golden@', :storageName,
    '.dfs.core.windows.net/', :carpeta_proyecto,
    '/fi_gd_mejor_juego_familia_decada',
    chr(39)
  );

  EXECUTE IMMEDIATE v_stmt;
END;


In [0]:
DESCRIBE CATALOG EXTENDED fi_videojuegos;

SHOW SCHEMAS IN fi_videojuegos;